In [110]:
import pandas as pd
import numpy as np

In [111]:
import pandas as pd

df = pd.read_csv("input.csv", encoding='utf-8')
submission_df = pd.read_csv("titanic_submission.csv", encoding='utf-8')

print(df.info())
df.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 791 entries, 0 to 790
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  791 non-null    int64  
 1   Survived     791 non-null    int64  
 2   Pclass       791 non-null    int64  
 3   Name         791 non-null    str    
 4   Sex          791 non-null    str    
 5   Age          614 non-null    float64
 6   SibSp        791 non-null    int64  
 7   Parch        791 non-null    int64  
 8   Ticket       791 non-null    str    
 9   Fare         791 non-null    float64
 10  Cabin        182 non-null    str    
 11  Embarked     789 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 74.3 KB
None


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [112]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['PassengerId','Survived','Name','Ticket','Cabin'])
y = df['Survived']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_tr.isnull().any())
print(X_tr.info())

Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked     True
dtype: bool
<class 'pandas.DataFrame'>
Index: 632 entries, 196 to 398
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    632 non-null    int64  
 1   Sex       632 non-null    str    
 2   Age       496 non-null    float64
 3   SibSp     632 non-null    int64  
 4   Parch     632 non-null    int64  
 5   Fare      632 non-null    float64
 6   Embarked  630 non-null    str    
dtypes: float64(2), int64(3), str(2)
memory usage: 39.5 KB
None


In [113]:
from sklearn.impute import SimpleImputer

num_cols = X_tr.select_dtypes(include='number').columns.tolist()
num_imputer = SimpleImputer(strategy='median')

X_tr[num_cols] = num_imputer.fit_transform(X_tr[num_cols])
X_te[num_cols] = num_imputer.transform(X_te[num_cols])


cat_cols = X_tr.select_dtypes(include=["object", "str"]).columns.tolist()
cat_imputer = SimpleImputer(strategy='most_frequent')

X_tr[cat_cols] = cat_imputer.fit_transform(X_tr[cat_cols])
X_te[cat_cols] = cat_imputer.transform(X_te[cat_cols])


In [114]:
from sklearn.preprocessing import OneHotEncoder

encoder_ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

tr_encoded = encoder_ohe.fit_transform(X_tr[cat_cols])
te_encoded = encoder_ohe.transform(X_te[cat_cols])

encoded_cols = encoder_ohe.get_feature_names_out(cat_cols)

tr_encoded = pd.DataFrame(tr_encoded, columns=encoded_cols, index=X_tr.index)
te_encoded = pd.DataFrame(te_encoded, columns=encoded_cols, index=X_te.index)

# 기존 범주형 컬럼 제거 + 인코딩 컬럼 추가
X_tr = pd.concat([X_tr.drop(columns=cat_cols), tr_encoded], axis=1)
X_te = pd.concat([X_te.drop(columns=cat_cols), te_encoded], axis=1)

tr_encoded

,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
196,0.0,1.0,0.0,0.0,1.0
254,1.0,0.0,0.0,0.0,1.0
366,1.0,0.0,0.0,0.0,1.0
287,1.0,0.0,1.0,0.0,0.0
158,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
430,1.0,0.0,0.0,0.0,1.0
772,0.0,1.0,0.0,0.0,1.0
240,1.0,0.0,0.0,1.0,0.0
306,1.0,0.0,0.0,0.0,1.0


In [115]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    oob_score=True,
    n_jobs=-1,
    random_state=101 
)

model_rf.fit(X_tr, y_tr)

# y_pred = model_rf.predict(X_te)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",101
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.

In [116]:
y_pred = model_rf.predict(X_te)


In [117]:
from sklearn.metrics import accuracy_score
accuracy_score(y_te, y_pred)

0.8553459119496856

In [118]:
# 1. 학습 데이터와 동일한 원본 컬럼 선택
X_submission = submission_df[X.columns].copy()

# 2. 숫자형 결측치 처리
X_submission[num_cols] = num_imputer.transform(
    X_submission[num_cols]
)

# 3. 범주형 결측치 처리
X_submission[cat_cols] = cat_imputer.transform(
    X_submission[cat_cols]
)

# 4. One-Hot Encoding
submission_encoded = encoder_ohe.transform(
    X_submission[cat_cols]
)

submission_encoded = pd.DataFrame(
    submission_encoded,
    columns=encoder_ohe.get_feature_names_out(cat_cols),
    index=X_submission.index
)

# 5. 기존 범주형 컬럼 제거 후 결합
X_submission = pd.concat([
    X_submission.drop(columns=cat_cols),
    submission_encoded
], axis=1)

# 6. 학습 데이터와 컬럼 순서 일치
X_submission = X_submission[X_tr.columns]

# 7. 예측
y_submission = model_rf.predict(X_submission)

# 8. CSV 저장
result = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': y_submission
})

result.to_csv("output.csv", index=False)
display(result.head())

,PassengerId,Survived
0,660,0
1,353,0
2,667,0
3,545,1
4,10,1
